## Diabetes Prediction 

### Importing the Dependencies

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import accuracy_score

### Data Collection and Analysis

### PIMA Diabetes Dataset

In [2]:
# load dataset
diabetes_dataset = pd.read_csv('diabetes-dataset.csv') 

In [3]:
# printing the first 5 rows of the dataset
diabetes_dataset.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,2,138,62,35,0,33.6,0.127,47,1
1,0,84,82,31,125,38.2,0.233,23,0
2,0,145,0,0,0,44.2,0.630,31,1
3,0,135,68,42,250,42.3,0.365,24,1
4,1,139,62,41,480,40.7,0.536,21,0


In [4]:
diabetes_dataset.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,3.703500,121.182500,69.145500,20.935000,80.254000,32.193000,0.470930,33.090500,0.342000
std,3.306063,32.068636,19.188315,16.103243,111.180534,8.149901,0.323553,11.786423,0.474498
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,63.500000,0.000000,0.000000,27.375000,0.244000,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,40.000000,32.300000,0.376000,29.000000,0.000000
75%,6.000000,141.000000,80.000000,32.000000,130.000000,36.800000,0.624000,40.000000,1.000000
max,17.000000,199.000000,122.000000,110.000000,744.000000,80.600000,2.420000,81.000000,1.000000


In [5]:
diabetes_dataset.shape

(2000, 9)

In [6]:
# handle missing values
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols:
    diabetes_dataset[col] = diabetes_dataset[col].replace(0, np.nan)
    diabetes_dataset[col] = diabetes_dataset[col].fillna(diabetes_dataset[col].median())

# after filling
print((diabetes_dataset[cols] == 0).sum())

Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


In [7]:
diabetes_dataset.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,2,138.0,62.0,35.0,126.0,33.6,0.127,47,1
1,0,84.0,82.0,31.0,125.0,38.2,0.233,23,0
2,0,145.0,72.0,29.0,126.0,44.2,0.630,31,1
3,0,135.0,68.0,42.0,250.0,42.3,0.365,24,1
4,1,139.0,62.0,41.0,480.0,40.7,0.536,21,0


In [8]:
diabetes_dataset['Outcome'].value_counts()

Outcome
0    1316
1     684
Name: count, dtype: int64

In [45]:
for col in diabetes_dataset.columns:
    print(f"{col} : min {min(diabetes_dataset[col])} and max {max(diabetes_dataset[col])}")

Pregnancies : min 0 and max 17
Glucose : min 44.0 and max 199.0
BloodPressure : min 24.0 and max 122.0
SkinThickness : min 7.0 and max 110.0
Insulin : min 14.0 and max 744.0
BMI : min 18.2 and max 80.6
DiabetesPedigreeFunction : min 0.078 and max 2.42
Age : min 21 and max 81
Outcome : min 0 and max 1


0 = Non-Diabetic <br/>
1 = Diabetic

In [9]:
diabetes_dataset.groupby('Outcome').mean()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Outcome,,,,,,,,
0,3.168693,111.208967,70.885258,27.897416,128.872340,31.182979,0.434676,31.081307
1,4.732456,142.595029,75.271930,31.833333,162.818713,35.462573,0.540681,36.956140


In [10]:
X = diabetes_dataset.drop(columns = ['Outcome'], axis=1)
Y = diabetes_dataset['Outcome']

In [11]:
print(X[1:6])

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
1            0     84.0           82.0           31.0    125.0  38.2   
2            0    145.0           72.0           29.0    126.0  44.2   
3            0    135.0           68.0           42.0    250.0  42.3   
4            1    139.0           62.0           41.0    480.0  40.7   
5            0    173.0           78.0           32.0    265.0  46.5   

   DiabetesPedigreeFunction  Age  
1                     0.233   23  
2                     0.630   31  
3                     0.365   24  
4                     0.536   21  
5                     1.159   58  


In [12]:
print(Y[1:6])

1    0
2    1
3    1
4    0
5    0
Name: Outcome, dtype: int64


### Data Standardization

In [13]:
scaler = StandardScaler()

In [14]:
scaler.fit(X)

StandardScaler()

In [15]:
standardized_data = scaler.transform(X)

In [16]:
print(standardized_data)

[[-0.5153943   0.52597447 -0.86930967 ...  0.13263038 -1.06324616
   1.18042417]
 [-1.12049474 -1.24288779  0.80477375 ...  0.77255042 -0.7355513
  -0.85632626]
 [-1.12049474  0.75527143 -0.03226796 ...  1.60722872  0.49175869
  -0.17740945]
 ...
 [ 0.69480658 -1.21013108  0.46995707 ... -0.20124094 -0.27492362
   0.75610116]
 [-1.12049474  0.23116409  3.14849054 ...  4.79291758 -0.46968566
  -0.60173245]
 [-0.5153943  -1.34115792 -0.03226796 ... -0.35426529  0.23516743
  -0.68659705]]


In [17]:
X = standardized_data
Y = diabetes_dataset['Outcome']

In [18]:
print(X[1:6])
print(Y[1:6])

[[-1.12049474 -1.24288779  0.80477375  0.1925422  -0.1898643   0.77255042
  -0.7355513  -0.85632626]
 [-1.12049474  0.75527143 -0.03226796 -0.02669173 -0.17760075  1.60722872
   0.49175869 -0.17740945]
 [-1.12049474  0.42770434 -0.36708464  1.39832882  1.34307963  1.34291392
  -0.32747846 -0.77146166]
 [-0.81794452  0.55873118 -0.86930967  1.28871185  4.16369648  1.12033304
   0.20116136 -1.02605546]
 [-1.12049474  1.67245927  0.46995707  0.30215916  1.52703291  1.92718873
   2.12714152  2.11393478]]
1    0
2    1
3    1
4    0
5    0
Name: Outcome, dtype: int64


### Train Test Split

In [19]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.2, stratify=Y, random_state=2)

In [20]:
print(X.shape, X_train.shape, X_test.shape)

(2000, 8) (1600, 8) (400, 8)


### Training the Model

#### Support vector machine/classifer

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn import svm

# define the parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto']
}

# grid search with 5-fold cross-validation
grid = GridSearchCV(
    estimator = svm.SVC(class_weight='balanced'),
    param_grid = param_grid,
    cv = 5,
    scoring = 'accuracy',
    n_jobs = -1
)

grid.fit(X_train, Y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)

Best parameters: {'C': 100, 'gamma': 'scale', 'kernel': 'rbf'}
Best cross-validation accuracy: 0.8856249999999999


In [34]:
# classifier = svm.SVC(kernel='linear')
# using best params we got ...
classifier = svm.SVC(kernel='rbf', C= 100, gamma='scale', class_weight='balanced')

In [35]:
#training the support vector Machine Classifier
classifier.fit(X_train, Y_train)

SVC(C=100, class_weight='balanced')

### Model Evaluation
#### Accuracy scores of train and test

In [37]:
# accuracy score on the training data
X_train_prediction = classifier.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [38]:
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.96625


In [39]:
# accuracy score on the test data
X_test_prediction = classifier.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [40]:
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9425


#### Making a Predictive System

In [28]:
#Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age

input_data = (2,166,80,35,90,33.8,0.58,50)
numpy_array = np.asarray(input_data)
data_reshaped = numpy_array.reshape(1,-1)

# standardize 
std_data = scaler.transform(data_reshaped)
print(std_data)

prediction = classifier.predict(std_data)
print(prediction)

if (prediction[0] == 0):
  print('The person is not diabetic')
else:
  print('The person is diabetic')

[[-0.5153943   1.44316231  0.63736541  0.63101006 -0.61908861  0.16045299
   0.33718564  1.43501797]]
[1]
The person is diabetic


C:\Users\saiha\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


#### Other models: Random Forest and Gradient Boosting

In [29]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42)

rf.fit(X_train, Y_train)
gb.fit(X_train, Y_train)

rf_pred = rf.predict(X_test)
gb_pred = gb.predict(X_test)

from sklearn.metrics import accuracy_score

print(f"Random Forest Accuracy: {accuracy_score(Y_test, rf_pred)}")
print(f"Gradient Boosting Accuracy: { accuracy_score(Y_test, gb_pred) }")

Random Forest Accuracy: 0.9725
Gradient Boosting Accuracy: 0.92


#### Just for reference (in general, the values ranges and results)
<table>
    <th>Factor</th>
    <th>Range</th>
    <th>Diabetic / Not Diabetic</th>
    <tr>
        <td>Pregnencies</td>
        <td>0–6</td>
        <td>Non Diabetic</td>
    </tr>
    <tr>
        <td>Pregnencies</td>
        <td>2–8</td>
        <td>Diabetic (Gestastional)</td>
    </tr>
    <tr>
        <td>Glucose</td>
        <td>85-120</td>
        <td>Non Diabetic</td>
    </tr>
    <tr>
        <td>Glucose</td>
        <td>130-180</td>
        <td>Diabetic</td>
    </tr>
    <tr>
        <td>BP</td>
        <td>65–85</td>
        <td>Non Diabetic</td>
    </tr>
    <tr>
        <td>BP</td>
        <td>70–90</td>
        <td>Diabetic</td>
    </tr>
    <tr>
        <td>Skin Thickness (Triceps)</td>
        <td>15-30</td>
        <td>Non Diabetic</td>
    </tr>
    <tr>
        <td>Skin Thickness (Triceps)</td>
        <td>20-40</td>
        <td>Diabetic</td>
    </tr>
    <tr>
        <td>Insulin</td>
        <td>0–150</td>
        <td>Non D</td>
    </tr>
    <tr>
        <td>Insulin</td><td>100-250</td><td>Diabetic</td>
    </tr>
    <tr>
        <td>BMI (Body Mass Index)</td><td>24-33</td><td>Non Diabetic</td>
    </tr>
    <tr>
        <td>BMI (Body Mass Index)</td><td>35–more</td><td>Diabetic</td>
    </tr>
    <tr>
        <td>Diabetes Pedigree (family history)</td><td>0.3–0.5</td><td>Non Diabetic</td>
    </tr>
    <tr>
        <td>Diabetes Pedigree(family history)</td><td>0.4–0.7</td><td>Diabetic</td>
    </tr>
    <tr>
        <td>Age</td><td>10-20</td><td>Non Diabetic</td>
    </tr>
    <tr>
        <td>Age</td><td>30 - older</td><td>Diabetic</td>
    </tr>
</table>